In [ ]:
# 2. Import libraries
import os
from pyspark.sql import SparkSession

# Fetch credentials passed from docker-compose
MINIO_USER = os.environ.get("MINIO_ROOT_USER")
MINIO_PASS = os.environ.get("MINIO_ROOT_PASSWORD")

print("Initializing Spark Session... Downloading Delta and AWS S3 dependencies...")

# Define required packages for Kafka, Delta, and S3
packages = [
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
    "io.delta:delta-spark_2.12:3.1.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

# Build the Spark Session
spark = SparkSession.builder \
    .appName("Kafka-to-MinIO-Lakehouse") \
    .master("local[*]") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_USER) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_PASS) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Containerized Spark Session successfully created!")

Initializing Spark Session... Downloading Delta and AWS S3 dependencies...


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import from_json, col

print("Defining schema and connecting to Kafka stream...")

# 1. Define the schema that matches our Faker generator output
schema = StructType([
    StructField("timestamp", DoubleType(), True),
    StructField("source_ip", StringType(), True),
    StructField("dest_ip", StringType(), True),
    StructField("source_port", IntegerType(), True),
    StructField("dest_port", IntegerType(), True),
    StructField("bytes", IntegerType(), True),
    StructField("is_attack", IntegerType(), True)
])

# 2. Read the stream from Kafka
# We use 'kafka:9092' because Spark and Kafka are on the same Docker network
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "network-traffic") \
    .option("startingOffsets", "latest") \
    .load()

# 3. Parse the JSON payload from Kafka's 'value' column
parsed_stream = kafka_stream.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# 4. Write the stream to MinIO as a Delta Lake table
# Checkpoints are mandatory for streaming to track what has been processed
S3_PATH = "s3a://network-logs/raw-traffic/"
CHECKPOINT_PATH = "s3a://network-logs/checkpoints/raw-traffic/"

print(f"Writing stream to Delta Lake at {S3_PATH}...")

streaming_query = parsed_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .start(S3_PATH)

print("Streaming query is actively running in the background!")

Defining schema and connecting to Kafka stream...
Writing stream to Delta Lake at s3a://network-logs/raw-traffic/...
Streaming query is actively running in the background!


In [3]:
streaming_query.stop()
print("Streaming query successfully stopped.")

Streaming query successfully stopped.


In [4]:
from pyspark.sql.functions import count, col

print("Loading historical batch data from Delta Lake...")

# Read the Delta table as a static batch DataFrame
batch_df = spark.read \
    .format("delta") \
    .load("s3a://network-logs/raw-traffic/")

# Show the schema and total record count
batch_df.printSchema()
total_records = batch_df.count()
print(f"Total records in Lakehouse: {total_records}")

# Group by the 'is_attack' column to see our class distribution
print("\nThreat Distribution:")
batch_df.groupBy("is_attack").agg(count("*").alias("count")).show()

# Show 5 sample records
print("Sample Data:")
batch_df.show(5)

Loading historical batch data from Delta Lake...
root
 |-- timestamp: double (nullable = true)
 |-- source_ip: string (nullable = true)
 |-- dest_ip: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- dest_port: integer (nullable = true)
 |-- bytes: integer (nullable = true)
 |-- is_attack: integer (nullable = true)

Total records in Lakehouse: 28783

Threat Distribution:
+---------+-----+
|is_attack|count|
+---------+-----+
|        1| 1430|
|        0|27353|
+---------+-----+

Sample Data:
+--------------------+--------------+---------------+-----------+---------+-----+---------+
|           timestamp|     source_ip|        dest_ip|source_port|dest_port|bytes|is_attack|
+--------------------+--------------+---------------+-----------+---------+-----+---------+
|1.7806079576603823E9| 35.185.48.136|   133.64.75.92|      14987|     8080| 1750|        0|
|1.7806079576718853E9|137.155.11.136|     98.36.2.98|      61251|       22|  654|        0|
|1.7806079576829066E